### Structured Output
Models can be requested to provide their response in a format matching given schema. This is useful for ensuring the output can be easily parsed.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [11]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")

#### Pydantic

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year movie was released")
    director: str = Field(description="Director of the movie")
    rating: float = Field(description="Movie rating out of 10")

In [13]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022A7EC4B380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022A7F00A270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [14]:
model.invoke("Provide details about the movie Interstellar")

AIMessage(content='<think>\nOkay, let\'s tackle this query about the movie "Interstellar." First, I need to recall the basic information about the film. Directed by Christopher Nolan, right? It came out in 2014. The main cast includes Matthew McConaughey, Anne Hathaway, and others. The genre is sci-fi, so I should mention that. The plot revolves around space exploration and wormholes, maybe time dilation? I think there\'s a part with a black hole called Gargantua.\n\nWait, the user wants details, so I should structure the answer properly. Let me start with the director, release year, and main cast. Then summarize the plot. I should explain the key themes like space exploration, time, and love. Maybe mention the scientific aspects, like the collaboration with Kip Thorne, the use of real equations for the visuals. The emotional elements are important too, like the relationship between Cooper and his daughter Murph. Also, the soundtrack by Hans Zimmer could be a point. Don\'t forget to no

In [17]:
response = model_with_structure.invoke("Provide details about the movie Interstellar")
response

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

In [18]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="Title of the movie")
    year: int = Field(..., description="year movie was released")
    director: str = Field(..., description="Director of movie")
    rating: float = Field(..., description="Movie rating out of 10")
    
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Proviode details about Revenant")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Revenant." Let me see what I need to do here. They want information, so I should use the Movie function they provided. The function requires title, year, director, and rating. I need to make sure I have all those details.\n\nFirst, the title is "Revenant." I should confirm the exact title. The director is Alejandro González Iñárritu. I think that\'s right. The year it was released was 2015. The rating, maybe around 8.0? I\'m not entirely sure, but I think that\'s accurate. Let me double-check those details in my mind. Yes, I remember "Revenant" was directed by Iñárritu, came out in 2015, and had a high rating. Alright, I\'ll structure the JSON with those parameters. Make sure all required fields are included and the types are correct. Title is a string, year is integer, director is a string, and rating is a number. Everything looks good. Let\'s put it all togethe

#### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Fight Club")
response

MovieDetails(title='Fight Club', year=1999, cast=[Actor(name='Brad Pitt', role='Tyler Durden'), Actor(name='Edward Norton', role='The Narrator'), Actor(name='Helena Bonham Carter', role='Marla Singer')], genres=['Drama', 'Thriller'], budget=63000000.0)

#### TypedDict

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, ..., "title of movie"]
    year: Annotated[int, ..., "year movie was released"]
    director: Annotated[str, ..., "director of the movie"]
    rating: Annotated[float, ..., "movie rating out of 10"]
    
model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("provide details about the movie Truman Show")
response

{'director': 'Peter Weir',
 'rating': 8.1,
 'title': 'The Truman Show',
 'year': 1998}

In [20]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

#### Dataclasses

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(description="name of person")
    email: str = Field(description="email address of person")
    phone: str = Field(description="phone number of person")
    
agent = create_agent(model=model, response_format=ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6b50bac4-3981-43c6-bd0b-d1719591b7ce'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is formatted as "(555) 123-4567". I should check if the phone number needs any formatting changes, but the tool\'s parameters just require a string, so the given format should be acceptable. \n\nI need to make sure all three fields are present. The user provided all three, so I can map them directly. The function requires name, email, and phone i

In [22]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [23]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str
    
agent = create_agent(model=model, response_format=ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')